In [16]:
import pandas as pd

pairs = pd.read_excel("conjoint_pairs_300_final.xlsx")

In [17]:
attrs = {
    "region": "Регион происхождения",
    "motivation": "Мотивация приезда",
    "education": "Уровень образования",
    "employer": "Тип работодателя",
    "gender": "Пол",
    "age": "Возраст",
    "occupation": "Планируемая сфера занятости",
    "language": "Владение русским языком",
    "politics": "Как страна происхождения мигранта относится к России",
    "appearance": "Внешний вид",
}
intro = "Ниже Вы увидите двух кандидатов"
question = "Если бы Вам нужно было выбрать между ними, кого из этих двух мигрантов Вы бы скорее предпочли видеть своим соседом по лестничной клетке?"

In [18]:
print(pairs.columns.tolist())

['pair_id', 'A_region', 'B_region', 'A_motivation', 'B_motivation', 'A_education', 'B_education', 'A_employer', 'B_employer', 'A_gender', 'B_gender', 'A_age', 'B_age', 'A_occupation', 'B_occupation', 'A_language', 'B_language', 'A_politics', 'B_politics', 'A_appearance', 'B_appearance']


In [19]:
def make_rows(row, side):
    result = ""
    for attr, label in attrs.items():
        value = row[f"{side}_{attr}"]
        result += f'<tr><td style="padding-bottom: 12px;"><strong>{label}:</strong> {value}</td></tr>'
    return result

In [20]:
def make_block(row):
    a_rows = make_rows(row, "A")
    b_rows = make_rows(row, "B")
    return f"""<p>{intro}</p>
<table style="width: 100%;">
  <tr>
    <td style="width: 50%; vertical-align: top; padding-right: 20px;">
      <p><strong>Кандидат А</strong></p>
      <table style="width: 100%;">{a_rows}</table>
    </td>
    <td style="width: 50%; vertical-align: top; padding-left: 20px;">
      <p><strong>Кандидат Б</strong></p>
      <table style="width: 100%;">{b_rows}</table>
    </td>
  </tr>
</table>
<p style="margin-top: 20px;"><strong>{question}</strong></p>"""


blocks = []
for i in range(len(pairs)):
    blocks.append(make_block(pairs.iloc[i]))


In [21]:
letters = {
    "region": "R",
    "motivation": "M",
    "education": "Ed",
    "employer": "Em",
    "gender": "G",
    "age": "A",
    "occupation": "O",
    "language": "L",
    "politics": "P",
    "appearance": "Ap",
}
codes = {}
for attr in attrs:
    unique = sorted(set(pairs[f"A_{attr}"].astype(str)) | set(pairs[f"B_{attr}"].astype(str)))
    codes[attr] = {v: i + 1 for i, v in enumerate(unique)}

def make_label(row):
    a = "".join(f"{letters[attr]}{codes[attr][str(row[f'A_{attr}'])]}" for attr in attrs)
    b = "".join(f"{letters[attr]}{codes[attr][str(row[f'B_{attr}'])]}" for attr in attrs)
    return f"{a} / {b}"

variable_names = [f"CJ{i + 1}" for i in range(len(pairs))]
labels = [make_label(pairs.iloc[i]) for i in range(len(pairs))]

print("Расшифровка кодов:")
for attr, mapping in codes.items():
    print(f"  {letters[attr]} = {attr}")
    for value, code in mapping.items():
        print(f"    {letters[attr]}{code} = {value}")

Расшифровка кодов:
  R = region
    R1 = Африка (Нигерия, ЮАР, Кения)
    R2 = Восточная Азия (Китай, Южная Корея, Монголия)
    R3 = Восточная Европа (Беларусь, Молдова, Украина)
    R4 = Западная Европа (Германия, Франция, Италия)
    R5 = Средняя Азия (Узбекистан, Таджикистан, Кыргызстан)
    R6 = Юго-Восточная Европа (Венгрия, Сербия, Болгария)
    R7 = Южный Кавказ (Армения, Азербайджан, Грузия)
  M = motivation
    M1 = Бегство от вооружённого конфликта
    M2 = Бегство от политических преследований
    M3 = Воссоединение с супругом(ой), ранее приехавшим(ей) в Россию
    M4 = Поиск работы
  Ed = education
    Ed1 = Высшее (университет)
    Ed2 = Среднее (школа)
    Ed3 = Среднее специальное (колледж)
  Em = employer
    Em1 = Государственная организация
    Em2 = Небольшая частная компания
  G = gender
    G1 = Женщина
    G2 = Мужчина
  A = age
    A1 = 21
    A2 = 22
    A3 = 23
    A4 = 45
    A5 = 46
    A6 = 47
    A7 = 61
    A8 = 62
    A9 = 63
  O = occupation
    O1 = Вр

In [22]:
with open("conjoint_pairs_final.txt", "w", encoding="utf-8") as f:
    for i, block in enumerate(blocks):
        f.write(f"Пара {i + 1}\n")
        f.write(f"Variable name: {variable_names[i]}\n")
        f.write(f"Label: {labels[i]}\n")
        f.write(block)
        f.write("\n\n")

# html для просмотра в браузере
with open("conjoint_pairs_final.html", "w", encoding="utf-8") as f:
    f.write("<html><head><meta charset='utf-8'></head><body>")
    for i, block in enumerate(blocks):
        f.write(f"<h3>Пара {i + 1}</h3>")
        f.write(block)
        f.write("<hr>")
    f.write("</body></html>")

In [23]:
mapping = pairs.copy()
mapping.insert(0, "variable_name", variable_names)
mapping.insert(1, "label", labels)
mapping.to_csv("cj_mapping_final.csv", index=False, encoding="utf-8-sig")
mapping.head()

,variable_name,label,pair_id,A_region,B_region,A_motivation,B_motivation,A_education,B_education,A_employer,...,A_age,B_age,A_occupation,B_occupation,A_language,B_language,A_politics,B_politics,A_appearance,B_appearance
0,CJ1,R3M2Ed3Em1G1A8O4L2P1Ap1 / R3M2Ed1Em2G1A8O4L2P1Ap1,1,"Восточная Европа (Беларусь, Молдова, Украина)","Восточная Европа (Беларусь, Молдова, Украина)",Бегство от политических преследований,Бегство от политических преследований,Среднее специальное (колледж),Высшее (университет),Государственная организация,...,62,62,Сфера услуг (общепит),Сфера услуг (общепит),Говорит свободно,Говорит свободно,Страна обычно голосует вместе с Россией в Гена...,Страна обычно голосует вместе с Россией в Гена...,В повседневной жизни носит обычную повседневну...,В повседневной жизни носит обычную повседневну...
1,CJ2,R1M4Ed1Em1G2A4O4L2P2Ap2 / R1M4Ed1Em2G1A2O4L1P1Ap1,2,"Африка (Нигерия, ЮАР, Кения)","Африка (Нигерия, ЮАР, Кения)",Поиск работы,Поиск работы,Высшее (университет),Высшее (университет),Государственная организация,...,45,22,Сфера услуг (общепит),Сфера услуг (общепит),Говорит свободно,Говорит плохо,Страна обычно голосует вместе с США в Генассам...,Страна обычно голосует вместе с Россией в Гена...,В повседневной жизни носит традиционную национ...,В повседневной жизни носит обычную повседневну...
2,CJ3,R3M1Ed3Em2G2A3O4L2P2Ap1 / R4M4Ed3Em1G2A2O4L2P2Ap1,3,"Восточная Европа (Беларусь, Молдова, Украина)","Западная Европа (Германия, Франция, Италия)",Бегство от вооружённого конфликта,Поиск работы,Среднее специальное (колледж),Среднее специальное (колледж),Небольшая частная компания,...,23,22,Сфера услуг (общепит),Сфера услуг (общепит),Говорит свободно,Говорит свободно,Страна обычно голосует вместе с США в Генассам...,Страна обычно голосует вместе с США в Генассам...,В повседневной жизни носит обычную повседневну...,В повседневной жизни носит обычную повседневну...
3,CJ4,R1M4Ed1Em2G2A6O3L1P2Ap1 / R1M3Ed1Em1G1A6O3L2P3Ap2,4,"Африка (Нигерия, ЮАР, Кения)","Африка (Нигерия, ЮАР, Кения)",Поиск работы,"Воссоединение с супругом(ой), ранее приехавшим...",Высшее (университет),Высшее (университет),Небольшая частная компания,...,47,47,Строитель,Строитель,Говорит плохо,Говорит свободно,Страна обычно голосует вместе с США в Генассам...,Страна помогает России обходить санкции (парал...,В повседневной жизни носит обычную повседневну...,В повседневной жизни носит традиционную национ...
4,CJ5,R6M4Ed2Em2G2A7O3L2P4Ap2 / R4M4Ed2Em1G2A7O3L2P4Ap1,5,"Юго-Восточная Европа (Венгрия, Сербия, Болгария)","Западная Европа (Германия, Франция, Италия)",Поиск работы,Поиск работы,Среднее (школа),Среднее (школа),Небольшая частная компания,...,61,61,Строитель,Строитель,Говорит свободно,Говорит свободно,Страна присоединилась к санкциям против России,Страна присоединилась к санкциям против России,В повседневной жизни носит традиционную национ...,В повседневной жизни носит обычную повседневну...
